## 1. Configurations

### 1-1. Import libraries

In [1]:
from    typing      import  Callable, Hashable, Sequence, Union
from    pathlib     import  Path
import  yaml
import  pickle

from    time            import  time
from    tqdm.notebook   import  trange, tqdm

import  numpy       as      np
import  torch
from    torch.utils.data    import  TensorDataset, DataLoader
import  pandas      as      pd

from    matplotlib.axes     import  Axes
from    matplotlib.figure   import  Figure
import  matplotlib.pyplot   as      plt
import  seaborn             as      sns

import  append_path
from    deep_numerical              import  utils
from    deep_numerical.utils        import  GaussianNormalizer, relative_error
from    deep_numerical              import  numerical   as  dn
from    deep_numerical.numerical    import  distribution    as  D
from    train_utils     import  load_data

from    helper_3D       import  *
DIMENSION = 3

dtype  = torch.float
device = torch.device("cuda:3")
BATCH_SIZE      = 100000
METRIC_ORDER    = 2.0

DPI = 1000
DPI_PLOT = 100
LINEWIDTH = 0.7
SAVEFIG_PATH = Path().cwd() / f"figures_{DIMENSION}D"
SAVEFIG_PATH.mkdir(exist_ok=True)

### 1-2. Load trained models and training histories

In [2]:
model_data      = load_model_data(f"paths_{DIMENSION}D.yaml")
models          = instantiate_models(model_data, device)
normalizers     = instantiate_normalizers(model_data, device)
train_histories = instantiate_histories(model_data)

[fno_param] The model is not included in the comparison.
[pfno_param] The model is not included in the comparison.
[prfno_param] The model is not included in the comparison.
[fno_kernel] The model is not included in the comparison.
[pfno_kernel] The model is not included in the comparison.
[prfno_kernel] The model is not included in the comparison.


### 1-3. Plot of training data

Print of the number of paramters in each model

In [3]:
n_params = {
    'Models': [], 'Parameters': [],
}
for k, v in models.items():
    n_params['Models'].append(k)
    n_params['Parameters'].append(v.count_parameters())
n_params = pd.DataFrame(n_params)
n_params.set_index('Models', inplace=True)
print(n_params)

             Parameters
Models                 
rfno_param       836769
rfno_kernel      836769


Plot of training history

In [4]:
axes: np.ndarray[Axes]
fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True, dpi=DPI)

# Plot the data loss
axes[0, 0].set_title('Data loss (Train)')
axes[1, 0].set_title('Data loss (Validation)')
for k, v in train_histories.items():
    axes[0, 0].plot(
        v['train_rel_error'],
        linewidth   = LINEWIDTH,
        label       = k+" (train)",
        **CONFIG__PLOT_OF_LOSS[k],
    )
    axes[1, 0].plot(
        v['val_rel_error'],
        linewidth   = LINEWIDTH,
        label       = k+" (val)",
        **CONFIG__PLOT_OF_LOSS[k],
    )

# Plot the conservation loss
axes[0, 1].set_title('Conservation loss (Train)')
axes[1, 1].set_title('Conservation loss (Validation)')
for k, v in train_histories.items():
    axes[0, 1].plot(
        v['train_rel_error_cons'],
        linewidth   = LINEWIDTH,
        label       = k+" (train)",
        **CONFIG__PLOT_OF_LOSS[k],
    )
    axes[1, 1].plot(
        v['val_rel_error_cons'],
        linewidth   = LINEWIDTH,
        label       = k+" (val)",
        **CONFIG__PLOT_OF_LOSS[k],
    )

for ax in axes.flatten():
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Relative $L^2$ error')
    ax.grid()
    ax.set_ylim(5e-5, 5e-2)
    ax.set_yscale('log')
    ax.legend()

plt.tight_layout()
plt.savefig(SAVEFIG_PATH/"train_histories.pdf", dpi=600)
plt.cla()   # clear the current axes
plt.clf()   # clear the current figure
plt.close() # closes the current figure

/tmp/ipykernel_282473/1772649688.py:49: UserWarning: Attempt to set non-positive ylim on a log-scaled axis will be ignored.
  plt.clf()   # clear the current figure


Plot of training history (`size_of_window=period_of_cosine_annealing`)

In [5]:
axes: np.ndarray[Axes]
fig, axes = plt.subplots(2, 2, figsize=(16, 12), sharex=True, sharey=True, dpi=DPI)
_size_of_window = 20
weight = torch.ones((1, 1, _size_of_window), device="cpu") / _size_of_window
func_mean: Callable[[torch.Tensor], torch.Tensor] = \
    lambda x: torch.nn.functional.conv1d(
        x.reshape(1, 1, -1), weight
    ).flatten()

# Plot the data loss
axes[0, 0].set_title('Data loss (Train)')
axes[1, 0].set_title('Data loss (Validation)')
for k, v in train_histories.items():
    axes[0, 0].plot(
        func_mean(v['train_rel_error']),
        linewidth   = LINEWIDTH,
        label       = k+" (train)",
        **CONFIG__PLOT_OF_LOSS[k],
    )
    axes[1, 0].plot(
        func_mean(v['val_rel_error']),
        linewidth   = LINEWIDTH,
        label       = k+" (val)",
        **CONFIG__PLOT_OF_LOSS[k],
    )

# Plot the conservation loss
axes[0, 1].set_title('Conservation loss (Train)')
axes[1, 1].set_title('Conservation loss (Validation)')
for k, v in train_histories.items():
    axes[0, 1].plot(
        func_mean(v['train_rel_error_cons']),
        linewidth   = LINEWIDTH,
        label      = k+" (train)",
        **CONFIG__PLOT_OF_LOSS[k],
    )
    axes[1, 1].plot(
        func_mean(v['val_rel_error_cons']),
        linewidth   = LINEWIDTH,
        label      = k+" (val)",
        **CONFIG__PLOT_OF_LOSS[k],
    )

for ax in axes.flatten():
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Relative $L^2$ error')
    ax.grid()
    ax.set_ylim(5e-5, 5e-2)
    ax.set_yscale('log')
    ax.legend()

plt.tight_layout()
plt.savefig(SAVEFIG_PATH/"train_histories__slided.pdf", dpi=600)
plt.cla()   # clear the current axes
plt.clf()   # clear the current figure
plt.close() # closes the current figure

/tmp/ipykernel_282473/2071403638.py:55: UserWarning: Attempt to set non-positive ylim on a log-scaled axis will be ignored.
  plt.clf()   # clear the current figure


## 2. Conduct test

In [6]:
# Test data
with open(f"config_test_{DIMENSION}D.yaml") as f:
    test_config = yaml.load(f, Loader=yaml.FullLoader)

_data = test_config['pde_dataset']
TEST_PATH       = Path(_data['path_test'])
TEST_RESOLUTION = RESOLUTION = _data['resolution']
EXP_SPEED       = _data['exp_speed']
TEST_PART       = _data['test_part']
NUM_TIME_STEPS  = _data['num_time_steps']
arr_t = 0.1 * torch.arange(NUM_TIME_STEPS)

In [9]:
test_raw_data: dict[str, torch.Tensor] = {}

_test_data, _test_info = load_data(r"/media/junseung/47a90e46-3a9d-467c-bbee-066752b68532/GWANGJAE/datasets_kinetic/Boltzmann_3D/biMaxwellian/coeff7.96e-02/__res032__split_for_20", 32, EXP_SPEED, [21], size_zfill=2)
V_MAX           = _test_info['v_max']
V_EXTENT        = (-V_MAX, V_MAX, V_MAX, -V_MAX)
WHERE_CLOSED    = _test_info['v_where_closed']
test_raw_data['data']   = _test_data['data'][:, :NUM_TIME_STEPS].to(device)
test_raw_data['params'] = _test_data['vhs_alpha'].to(device)
test_gen_time = _test_info['elapsed_time']
# test_raw_data = augment_data_3D(test_raw_data)
TEST_SIZE   = test_raw_data['data'].shape[0]
del(_test_data, _test_info)
print(f"The size of the test dataset >>>", TEST_SIZE, sep=' ')
print(f"The shape of the test dataset >>>", test_raw_data['data'].shape, test_raw_data['params'].shape, sep=' ')
print(f"Time for generateing the test dataset >>> {int(test_gen_time)} seconds")

test_arg_params:    dict[int, torch.LongTensor] = {}
params_unique = list(test_raw_data['params'].unique())
params_unique.sort()
params_unique = tuple(params_unique)
for idx, p in enumerate(params_unique):
    test_arg_params[idx] = torch.argwhere(test_raw_data['params'].flatten()==p).flatten()

The size of the test dataset >>> 100
The shape of the test dataset >>> torch.Size([100, 51, 32, 32, 32, 1]) torch.Size([100, 1])
Time for generateing the test dataset >>> 799 seconds


In [ ]:
print(f"Construct a test dataset for each model.")
test_datasets: dict[str, TensorDataset] = {
    k: construct_datasets(
        k,
        test_raw_data['data'], test_raw_data['params'], normalizers[k],
        V_MAX, res_params=10,
    ) for k in models.keys()
}

print(f"Construct a test dataloader for each model.")
test_dataloaders: dict[str, DataLoader] = {
    k: DataLoader(v, batch_size=BATCH_SIZE, shuffle=False)
    for k, v in test_datasets.items()
}

Construct a test dataset for each model.
Construct a test dataloader for each model.


In [ ]:
print("Test the models")
print("* Initialize the storages of the predictions and the relative errors")
targets:        torch.Tensor = test_raw_data['data'].clone()
"""Saves *decoded* target data."""
pred_lists:     dict[str, Union[torch.Tensor, list[torch.Tensor]]]   = {}
"""Saves *decoded* predictions."""
rel_errors:     dict[str, Union[torch.Tensor, list[torch.Tensor]]]   = {}
"""Saves the relative errors between the *decoded* predictions and targets."""
inference_times:    dict[str, float] = {}
for k_model in models.keys():
    _normalizer     = normalizers[k_model]['data']
    _encoded_traj, _ = test_datasets[k_model][:]
    _encoded_traj = _encoded_traj[:, 0]
    pred_lists[k_model]     = [ _normalizer.decode(_encoded_traj).cpu() ]
    rel_errors[k_model]     = [ torch.zeros((TEST_SIZE,), device=device).cpu() ]
    inference_times[k_model]  = 0.0

print("* Start evaluating the models")
for k_model, model in models.items():
    print(f"  - Evaluating [{k_model}]...")
    model.eval()
    normalizers[k_model]['data'].to(device)
    normalizers[k_model]['params'].to(device)
    inference_times[k_model] = time()

    with torch.no_grad():
        traj:   torch.Tensor
        params: torch.Tensor
        for traj, params in test_dataloaders[k_model]:
            traj    = traj.to(device)
            params  = params.to(device)
            pred    = traj[:, 0]
            for idx in trange(NUM_TIME_STEPS - 1):
                # Time-marching (in the scaled space)
                pred = model.forward(pred, params)
                # Descale data
                pred_decoded    = normalizers[k_model]['data'].decode(pred)
                target_decoded  = targets[:, idx+1]
                # Compute the data loss
                _test_rel_error = relative_error(pred_decoded, target_decoded, p=METRIC_ORDER)
                # Save data
                pred_lists[k_model].append(pred_decoded.cpu())
                rel_errors[k_model].append(_test_rel_error.cpu())

    inference_times[k_model] = time() - inference_times[k_model]
    pred_lists[k_model] = torch.stack(pred_lists[k_model], dim=1)
    rel_errors[k_model] = torch.stack(rel_errors[k_model], dim=1)

print("Done.")

Test the models
* Initialize the storages of the predictions and the relative errors


* Start evaluating the models
  - Evaluating [rfno_param]...


  0%|          | 0/50 [00:00<?, ?it/s]

  - Evaluating [rfno_kernel]...


  0%|          | 0/50 [00:00<?, ?it/s]

Done.


In [ ]:
fig = plt.figure(figsize=(10, 6), dpi=DPI)
fig.suptitle(f"Average relative error ($p={METRIC_ORDER}$)")
for k in models.keys():
    _mean   = rel_errors[k].mean(dim=0)
    _std    = rel_errors[k].std(dim=0)
    _z = 1.0
    _lower  = torch.where(_mean-_z*_std>=0, _mean-_z*_std, 0)
    _upper  = _mean+_z*_std
    plt.plot(arr_t.cpu(), _mean, label=k, **CONFIG__PLOT_OF_LOSS[k])
    plt.fill_between(arr_t.cpu(), _lower, _upper, alpha=0.1, color=CONFIG__PLOT_OF_LOSS[k]['color'])
plt.legend()
plt.xlabel("$t$")
plt.yscale('log')
plt.ylim(5e-4, 1e-1)
plt.grid()
plt.tight_layout()
plt.savefig(SAVEFIG_PATH/"test_rel_error.pdf", dpi=DPI)
plt.cla()   # clear the current axes
plt.clf()   # clear the current figure
plt.close() # closes the current figure

In [ ]:
__title = "[ Training time per epoch | Inference time ]"
print(__title)
train_time_per_epoch = {
    k: v['train_time'] / len(v['train_rel_error'])
    for k, v in train_histories.items()
}
__elapsed_times = {
    'Models': [],
    'Training time per epoch': [],
    'Inference time': [],
}
for k in models.keys():
    __elapsed_times['Models'].append(k)
    __elapsed_times['Training time per epoch'].append(train_time_per_epoch[k])
    __elapsed_times['Inference time'].append(inference_times[k])
elapsed_times = pd.DataFrame(__elapsed_times)
elapsed_times.set_index('Models', inplace=True)
print(elapsed_times)
elapsed_times.plot.bar(edgecolor='black', figsize=(8, 5), width=0.4, rot=45, grid=True)
plt.suptitle(__title, fontsize=16)
plt.ylabel("Time (seconds)")
plt.tight_layout()
plt.savefig(SAVEFIG_PATH/"train_inference_time.pdf", dpi=DPI)
plt.cla()   # clear the current axes
plt.clf()   # clear the current figure
plt.close() # closes the current figure

[ Training time per epoch | Inference time ]
             Training time per epoch  Inference time
Models                                              
rfno_param                523.312839        8.221535
rfno_kernel               524.417401        8.194565


In [ ]:
selected_key_1 = K__RFNO_KERNEL
selected_key_2 = K__RFNO_PARAM
colormap = ['red', 'orange', 'yellow', 'green', 'blue']
plt.figure(figsize=(8, 5), dpi=DPI)
plt.suptitle(rf"Relative errors for [{selected_key_1}] and [{selected_key_2}] ($p={METRIC_ORDER}$)")
for order, mask in test_arg_params.items():
    mask = mask.cpu()
    p = params_unique[order]
    _mean_1     = rel_errors[selected_key_1][mask].mean(dim=0)
    _std_1      = rel_errors[selected_key_1][mask].std(dim=0)
    _lower_1    = torch.where(_mean_1-_std_1>=0, _mean_1-_std_1, 0)
    _upper_1    = _mean_1+_std_1
    plt.plot(
        arr_t.cpu(), _mean_1,
        linewidth=1, linestyle='dotted', color=colormap[order],
        label=rf"[{selected_key_1}] $\alpha={p:.2f}$"
    )
    # plt.fill_between(arr_t.cpu(), _lower_1, _upper_1, alpha=0.2, color=colormap[order])
    _mean_2     = rel_errors[selected_key_2][mask].mean(dim=0)
    _std_2      = rel_errors[selected_key_2][mask].std(dim=0)
    _lower_2    = torch.where(_mean_2-_std_2>=0, _mean_2-_std_2, 0)
    _upper_2    = _mean_2+_std_2
    plt.plot(
        arr_t.cpu(), _mean_2,
        linewidth=1, linestyle='solid', color=colormap[order],
        label=rf"[{selected_key_2}] $\alpha={p:.2f}$"
    )
    # plt.fill_between(arr_t.cpu(), _lower_2, _upper_2, alpha=0.2, color=colormap[order])
plt.xlabel("$t$")
plt.yscale('log')
plt.ylim((1e-4, 1e+0))
plt.grid()
plt.legend()
plt.tight_layout()
plt.savefig(SAVEFIG_PATH/"test_comparison.pdf", dpi=DPI)
plt.cla()   # clear the current axes
plt.clf()   # clear the current figure
plt.close() # closes the current figure

In [ ]:
for k in models.keys():
    print(f"[ {k} ]")
    print(f"- Inference time: {inference_times[k]:.2f} seconds")
    print(f"- Relative error at the terminal time (average, std_dev): {rel_errors[k][:, -1].mean().item():.2e}, {rel_errors[k][:, -1].std().item():.2e}")
    print(f"- Maximum relative error at the terminal time: {rel_errors[k][:, -1].max().item():.2e}")
    print(f"- Maximum relative error during inference: {rel_errors[k].max().item():.2e}")
    print('-' * 50)

[ rfno_param ]
- Inference time: 8.22 seconds
- Relative error at the terminal time (average, std_dev): 4.13e-03, 2.04e-03
- Maximum relative error at the terminal time: 8.98e-03
- Maximum relative error during inference: 8.98e-03
--------------------------------------------------
[ rfno_kernel ]
- Inference time: 8.19 seconds
- Relative error at the terminal time (average, std_dev): 4.21e-03, 1.27e-03
- Maximum relative error at the terminal time: 6.34e-03
- Maximum relative error during inference: 6.34e-03
--------------------------------------------------


In [ ]:
terminal_errors = pd.DataFrame(
    {
        k: rel_errors[k][:, -1]
        for k in models.keys()
    }
)
plt.title(f"Terminal relative errors ($p={METRIC_ORDER}$)")
terminal_errors.boxplot(figsize=(8, 5), rot=45, grid=True)
plt.yscale('log')
plt.tight_layout()
plt.savefig(SAVEFIG_PATH/"terminal_rel_error.pdf", dpi=DPI)
plt.cla()   # clear the current axes
plt.clf()   # clear the current figure
plt.close() # closes the current figure

End of file